## This section covers the determination of which sequences from Logan are "novel" e.g., less than 90% nucleotide identity to anything on blastn
1. Following the blastn run, determination of which are novel
2. retrieve amino acid sequences of those contigs, cluster and run msa + tree 
3. visualize tree in R, including attaching metadata and labelling for comprehensive view
4. generate 1-3 example hits, that have interesting features or are good model hits

In [2]:
%load_ext rpy2.ipython
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

palms sweaty, knees weak,

In [11]:
%%R
library(tidyverse)
#P1, determination of which are novel
###for novelty search
feb_7_L1_nuc_hmmer_env <- read.table("feb_7_L1_nuc_hmmer_env.tsv", sep = "\t", header = F)
#calculate qcov
feb_7_L1_nuc_hmmer_env$qcov <- abs(feb_7_L1_nuc_hmmer_env$V2 - feb_7_L1_nuc_hmmer_env$V3)/feb_7_L1_nuc_hmmer_env$V4
#evals fil 
feb_7_L1_nuc_hmmer_env_fil <- filter(feb_7_L1_nuc_hmmer_env, V10 < 0.0001)
feb_7_L1_nuc_hmmer_env_fil_low_conf <- filter(feb_7_L1_nuc_hmmer_env, V10 > 0.0001)


#look for those with highest % iden first in confident hits
feb_7_L1_nuc_hmmer_env_sliced <- feb_7_L1_nuc_hmmer_env_fil %>% group_by(V1) %>% slice_max(n = 1, V9)
#under 90% iden
feb_7_L1_nuc_hmmer_env_sliced_90 <- filter(feb_7_L1_nuc_hmmer_env_sliced, V9 < 90)
feb_7_L1_nuc_hmmer_env_sliced_90 = feb_7_L1_nuc_hmmer_env_sliced_90[!duplicated(feb_7_L1_nuc_hmmer_env_sliced_90$V1),]

feb_7_L1_nuc_hmmer_env_less_90_nt <- unique(feb_7_L1_nuc_hmmer_env_sliced_90$V1)
# feb_7_L1_nuc_hmmer_env_qcov_less50 <- filter(feb_7_L1_nuc_hmmer_env_qcov_less40, !V1 %in% feb_7_L1_nuc_hmmer_env_less_90_nt)

write.table(feb_7_L1_nuc_hmmer_env_less_90_nt, "feb_7_L1_nuc_hmmer_env_less_90_nt.txt", quote = F, col.names = F, row.names = F)

##some are not here!
#length(unique(feb_7_L1_nuc_hmmer_env_sliced$V1))
# 859 of 942, so 83 are unaccounted for

#create list of the ones that were hit, in general
hit_list <- unique(feb_7_L1_nuc_hmmer_env_sliced$V1)
write.table(hit_list, "hit_list.txt", quote = F, col.names = F, row.names = F)

#look for low conf hits that were not represented in either novel already, or other filter set
feb_7_L1_nuc_hmmer_env_fil_low_conf_hits <- filter(feb_7_L1_nuc_hmmer_env_fil_low_conf, !V1 %in% feb_7_L1_nuc_hmmer_env_less_90_nt)
feb_7_L1_nuc_hmmer_env_fil_low_conf_hits <- filter(feb_7_L1_nuc_hmmer_env_fil_low_conf_hits, !V1 %in% feb_7_L1_nuc_hmmer_env_fil$V1)

feb_7_L1_nuc_hmmer_env_fil_low_conf_hits_list <- unique(feb_7_L1_nuc_hmmer_env_fil_low_conf_hits$V1)

write.table(feb_7_L1_nuc_hmmer_env_fil_low_conf_hits_list, "feb_7_L1_nuc_hmmer_env_fil_low_conf_hits_list.txt", quote = F, col.names = F, row.names = F)

In [37]:
%%bash
#P1, determination of which are novel, continued
##look for those that were inputs, but had no matches against the blastn database
seqkit grep -v -f hit_list.txt feb_7_L1_nuc_hmmer_env_sorted_centroids.fa | grep ">" | wc -l
#83
#checks out

#print to file
seqkit grep -v -f hit_list.txt feb_7_L1_nuc_hmmer_env_sorted_centroids.fa > no_blastn_hits.fa

#extract those that are novel next
seqkit grep -f "/home/rnalab/js/pv_3/feb_7_L1_nuc_hmmer_env_less_90_nt.txt" feb_7_L1_nuc_hmmer_env_sorted_centroids.fa > feb_7_L1_nuc_hmmer_env_novel.fa
seqkit grep -f "/home/rnalab/js/pv_3/feb_7_L1_nuc_hmmer_env_fil_low_conf_hits_list.txt" feb_7_L1_nuc_hmmer_env_sorted_centroids.fa > feb_7_L1_nuc_hmmer_env_novel_low_conf.fa

#cat for full list of novel seqs
cat feb_7_L1_nuc_hmmer_env_novel.fa no_blastn_hits.fa feb_7_L1_nuc_hmmer_env_novel_low_conf.fa > feb_7_L1_nuc_hmmer_env_all_novel.fa
head feb_7_L1_nuc_hmmer_env_novel

bash: line 3: seqkit: command not found


0


bash: line 8: seqkit: command not found
bash: line 11: seqkit: command not found
bash: line 12: seqkit: command not found
head: cannot open 'feb_7_L1_nuc_hmmer_env_novel' for reading: No such file or directory


CalledProcessError: Command 'b'#P1, determination of which are novel, continued\n##look for those that were inputs, but had no matches against the blastn database\nseqkit grep -v -f hit_list.txt feb_7_L1_nuc_hmmer_env_sorted_centroids.fa | grep ">" | wc -l\n#83\n#checks out\n\n#print to file\nseqkit grep -v -f hit_list.txt feb_7_L1_nuc_hmmer_env_sorted_centroids.fa > no_blastn_hits.fa\n\n#extract those that are novel next\nseqkit grep -f "/home/rnalab/js/pv_3/feb_7_L1_nuc_hmmer_env_less_90_nt.txt" feb_7_L1_nuc_hmmer_env_sorted_centroids.fa > feb_7_L1_nuc_hmmer_env_novel.fa\nseqkit grep -f "/home/rnalab/js/pv_3/feb_7_L1_nuc_hmmer_env_fil_low_conf_hits_list.txt" feb_7_L1_nuc_hmmer_env_sorted_centroids.fa > feb_7_L1_nuc_hmmer_env_novel_low_conf.fa\n\n#cat for full list of novel seqs\ncat feb_7_L1_nuc_hmmer_env_novel.fa no_blastn_hits.fa feb_7_L1_nuc_hmmer_env_novel_low_conf.fa > feb_7_L1_nuc_hmmer_env_all_novel.fa\nhead feb_7_L1_nuc_hmmer_env_novel\n'' returned non-zero exit status 1.

In [ ]:
%%bash
#P2, retrieve amino acid sequences of those contigs, cluster and run msa + tree 
#transeq does not play well with colons
sed -i.bak 's/:/__/' "/home/rnalab/js/pv_3/feb_7_L1_nuc_hmmer_env_all_novel.fa"

#translate only in frame 1
transeq "/home/rnalab/js/pv_3/feb_7_L1_nuc_hmmer_env_all_novel.fa" feb_7_L1_nuc_hmmer_env_novel.aa -frame=1
#cluster at 90% aa, in preparation for "genus-level" tree

seqkit sort -l -r feb_7_L1_nuc_hmmer_env_novel.aa > feb_7_L1_nuc_hmmer_env_novel_sort.aa
usearch --cluster_smallmem feb_7_L1_nuc_hmmer_env_novel_sort.aa -id 0.90 -centroids feb_7_L1_nuc_hmmer_env_novel_sort_centroids_90.fa -uc feb_7_L1_nuc_hmmer_env_novel_sort_clusters_90.uc

# usearch v11.0.667_i86linux32, 4.0Gb RAM (214Gb total), 272 cores
# (C) Copyright 2013-18 Robert C. Edgar, all rights reserved.
# https://drive5.com/usearch

# License: personal use only

# 00:01 91Mb    100.0% 234 clusters, max size 2, avg 1.0
# 00:02 91Mb    100.0% Writing centroids to feb_7_L1_nuc_hmmer_env_novel_sort_centroids_90.fa

#       Seqs  240
#   Clusters  234
#   Max size  2
#   Avg size  1.0
#   Min size  1
# Singletons  228, 95.0% of seqs, 97.4% of clusters
#    Max mem  91Mb
#       Time  1.00s
# Throughput  240.0 seqs/sec.


#compare with rest of the PVs
#see ncbi char for generation of ncbi centroids
cat "/home/rnalab/js/ncbi_character/p123_ncbi_JR_centroids.aa" "/home/rnalab/js/pv_3/feb_7_L1_nuc_hmmer_env_novel_sort_centroids_90.fa" > feb_7_L1_ncbi_and_novel.aa

#do sequence alignment
muscle5.1.linux_intel64 -super5 "/home/rnalab/js/pv_3/feb_7_L1_ncbi_and_novel.aa" -output feb_7_L1_ncbi_and_novel.aln
# Input: 1039 seqs, length avg 398 max 483

# 00:00 4.5Mb   100.0% Derep 1035 uniques, 3 dupes
# 00:00 4.9Mb  CPU has 272 cores, defaulting to 20 threads
# 00:14 14Mb      2.0% UCLUST 1036 seqs EE<0.01, 20 centroids, 0 members
=
#dupes are likely caused by difference at nt level, but not aa level, acceptable for visualization purposes

#iqtree, with 1k BS
"/home/rnalab/js/logan_2/iqtree-2.3.6-Linux-intel/bin/iqtree2" -s feb_7_L1_ncbi_and_novel.aln -B 1000



In [6]:
%%R
# Bioconductor version
# install.packages("BiocManager", dependencies=TRUE, repos='http://cran.rstudio.com/')
# library(BiocManager)
# BiocManager::install("ggtree")

# # github version
# devtools::install_github("YuLab-SMU/ggtree")
# library(ggtree)

# install.packages("Polychrome", dependencies=TRUE, repos='http://cran.rstudio.com/')



* installing *source* package ‘Polychrome’ ...
** package ‘Polychrome’ successfully unpacked and MD5 sums checked
** using staged installation
** R
** data
** inst
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
** building package indices
** installing vignettes
** testing if installed package can be loaded from temporary location
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (Polychrome)


trying URL 'http://cran.rstudio.com/src/contrib/Polychrome_1.5.1.tar.gz'
Content type 'application/x-gzip' length 615879 bytes (601 KB)
downloaded 601 KB


The downloaded source packages are in
	‘/tmp/Rtmpk1fEds/downloaded_packages’
Updating HTML index of packages in '.Library'
Making 'packages.html' ... done


In [10]:
%%R
library(tidyverse)
library(Polychrome)
library(ggtree)
#to visualize the tree properly
ncbi_and_novel_tree <- read.tree("2025.02.18_placeholder.nhx")
#bootstrap value is saved as "label" in the tree 
ggtree(ncbi_and_novel_tree) + geom_tiplab() + geom_text(aes(label=label), hjust=-.3)

#to get the species labels for the tips, split into pave and SRA samples
tip_labs_ncbi_novel <- as.data.frame(ncbi_and_novel_tree[["tip.label"]])
colnames(tip_labs_ncbi_novel) <- c("ncbi_and_novel_tip")

#deal with SRA stuff, separate out for a clean input list
SRA_ncbi_and_novel <- as.data.frame(grep("[SED][R]{2}", tip_labs_ncbi_novel$ncbi_and_novel_tip, value = T ))
colnames(SRA_ncbi_and_novel) <- c('ncbi_and_novel_SRA')
SRA_ncbi_and_novel$ncbi_and_novel_SRA_info <- SRA_ncbi_and_novel$ncbi_and_novel_SRA
SRA_ncbi_and_novel <- SRA_ncbi_and_novel %>% separate(ncbi_and_novel_SRA, into = c("library", "info"), sep="_")
#grep with this
write.table(SRA_ncbi_and_novel$library, "ncbi_and_novel_SRA.txt", sep = "/t", col.names = F, row.names = F, quote = F)


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ tidyr::expand() masks ggtree::expand()
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In addition: Warning message:
Expected 2 pieces. Additional pieces discarded in 234 rows [1, 2, 3, 4, 5, 6,
7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, ...]. 


In [ ]:
%%bash
#search for correct accessions in the sra SQL query
zstd -dc "/home/rnalab/js/sra_taxid.csv.zst" | grep -f "/home/rnalab/js/pv_3/2025.02.18.ncbi_and_novel_SRA.txt" | awk 'BEGIN {FS=",";OFS="\t"} {print $1, $6}' | sed -e 's/"//g' > ncbi_and_novel_sra_info.txt

In [11]:
%%R
#using sra metadata file, grep two columns that are the sra library, and associated host id
#what this means is, get the list of novel libraries from above, input into https://www.ncbi.nlm.nih.gov/sites/batchentrez and download metadata from webportal
#save as tab delim file
SRA_entrez_nn <- read.table("ncbi_and_novel_sra_info.txt", sep = "\t", header = F)
#clean up a little
colnames(SRA_entrez_nn) <- c("library", "species")
SRA_entrez_nn <- SRA_entrez_nn[!duplicated(SRA_entrez_nn), ]

#annotate information on to the sra-only dataframe
SRA_merge <- left_join(SRA_ncbi_and_novel, SRA_entrez_nn, by = "library")
SRA_merge <- select(SRA_merge, ncbi_and_novel_SRA_info, species)

#repeat process for ncbi sequences
ncbi_tips <- filter(tip_labs_ncbi_novel, !ncbi_and_novel_tip %in% SRA_ncbi_and_novel$ncbi_and_novel_SRA_info)
ncbi_tips$ncbi_acc <- sub("-.*", "", ncbi_tips$ncbi_and_novel_tip)
ncbi_tips$ncbi_acc <- as.character(ncbi_tips$ncbi_acc)
write.table(ncbi_tips$ncbi_acc, "ncbi_tips.txt", sep = "/t", col.names = F, row.names = F, quote = F)


In [ ]:
%%bash
#download PVs from NCBI virus again (2024.09.21 ncbi_host.fasta)
#in bash, grep
grep ">" "/home/rnalab/js/ncbi_character/2024.09.21 ncbi_host.fasta" | awk 'BEGIN {FS="|";OFS="\t"} {print $1, $3}' | sed -e 's/>//' -e 's/\..*\t/\t/' > ncbi_info.txt
#and move this file back to R

In [12]:
%%R
#read in NCBI host file
ncbi_nn <- read.table("ncbi_info.txt", sep = "\t", header = F)
colnames(ncbi_nn) <- c('ncbi_acc', 'host') 
ncbi_nn$host <- as.character(ncbi_nn$host)
ncbi_nn$ncbi_acc <- as.character(ncbi_nn$ncbi_acc)
ncbi_nn$ncbi_acc <- stringr::str_trim(ncbi_nn$ncbi_acc)

ncbi_merge <- left_join(ncbi_tips, ncbi_nn, by = "ncbi_acc")

ncbi_host <- subset(ncbi_merge, trimws(host) !="")
ncbi_nohost <- subset(ncbi_merge, trimws(host) == "")
#get ones that didnt have host annotation
write.table(ncbi_nohost$ncbi_acc, "ncbi_nh.txt", sep = "/t", col.names = F, row.names = F, quote = F)

In [ ]:
%%bash
#in bash, grep
#make do with species demarcation
grep ">" "/home/rnalab/js/ncbi_character/2024.09.21 ncbi_host.fasta" | grep -w -f "/home/rnalab/js/pv_3/ncbi_nh.txt"  | awk 'BEGIN {FS="|";OFS="\t"} {print $1, $4}' | sed -e 's/>//' -e 's/\..*\t/\t/' > ncbi_info_fill.txt


In [14]:
%%R
#fill with next best thing, species
ncbi_nh <- read.table("ncbi_info_fill.txt", sep = "\t", header = F)
colnames(ncbi_nh) <- c('ncbi_acc', 'host') 

In [16]:
%%R
#trim whitespace and such
ncbi_nh$ncbi_acc <- stringr::str_trim(ncbi_nh$ncbi_acc)

ncbi_nohost$host <- NULL
ncbi_merge_nohost <- left_join(ncbi_nohost, ncbi_nh, by = "ncbi_acc")

ncbi_novel_rbind <- rbind(ncbi_merge_nohost, ncbi_host)

In [17]:
%%R
#combine the two labels
SRA_merge_info <- select(SRA_merge, ncbi_and_novel_SRA_info, species)
colnames(SRA_merge_info) <- c('Newick_label', 'host') 
SRA_merge_info_merge2 <- SRA_merge_info
colnames(SRA_merge_info_merge2) <- c('Newick_label', 'status')
SRA_merge_info_merge3 <- SRA_merge_info_merge2
colnames(SRA_merge_info_merge3) <- c('Newick_label', 'status') 

In [20]:
%%R
#merge dataframes to finally go in
ncbi_novel_rbind_info <- select(ncbi_novel_rbind, ncbi_and_novel_tip, host)
colnames(ncbi_novel_rbind_info) <- c('Newick_label', 'species') 
ncbi_novel_rbind_info_merge <- ncbi_novel_rbind_info
colnames(ncbi_novel_rbind_info_merge) <- c('Newick_label', 'status') 

In [ ]:
write.table(ncbi_novel_rbind_info_merge, "OUT_ncbi_tags_manual.txt", sep = "/t", col.names = F, row.names = F, quote = F)
#for those that are species, go back and manually annotate by searching up the PV species (if the name is non informative, eg. alphapapillomavirus 49)

In [34]:
%%R
library(tidyverse)
#read back in for complete ncbi stuff
ncbi_tags_manual <- read.table("ncbi_tags_manual.txt", sep = "\t", header = T)
colnames(ncbi_tags_manual) <- c('Newick_label', 'status') 
ncbi_tags_manual_and_sra <- rbind(SRA_merge_info_merge3, ncbi_tags_manual)
colnames(ncbi_tags_manual) <- c('Newick_label', 'uhhh') 


In [35]:
%%R
#finally, bind with the SRA stuff
known_ncbi_and_novel_all <- rbind(ncbi_novel_rbind_info_merge, SRA_merge_info_merge2)
colnames(known_ncbi_and_novel_all) <- c('Newick_label', 'status')

#add this information back to the newick tree
p <- ggtree(ncbi_and_novel_tree, aes(color = status)) %<+% ncbi_tags_manual_and_sra

P50 = createPalette(251,  c("#ff0000", "#00ff00", "#0000ff"))

names(P50) <- NULL

#v2 of graph with colored branches and various other values
pp <- p + geom_tippoint() + 
  # geom_text(aes(label = species), size = 4, hjust = -0.05, color = "black") +
  # geom_tiplab(size = 2, hjust = -2.5, color = "black") +
  # geom_text(aes(label=label), hjust=-0.5, size = 2, color = "black") +
  # geom_nodelab(label = ncbi_and_novel_tree$node.label, geom = 'text', size = 1.5) +
  # scale_color_manual(values = P50) +
  theme(legend.position= "none")+ 
  scale_color_manual(values = P50)

# known_ncbi_and_novel <- as.data.frame(grep("ncbi_and_novelase.|MGY", tip_labs_ncbi_novel$ncbi_and_novel_tip, value = T ))
# known_ncbi_and_novel$species <- c("known")
# colnames(known_ncbi_and_novel) <- c('Newick_label', 'status') 

ppp <- pp %<+% SRA_merge_info
pppp <- ppp + geom_tippoint() + geom_text(aes(label = host), size = 4, hjust = -0.05, color = "black", fontface = "bold") +  geom_tiplab(size = 2, hjust = -1.5, color = "white", alpha = 0) 

ppppp <- pppp %<+% ncbi_tags_manual
pppppp <- ppppp + geom_tippoint() + geom_text(aes(label = uhhh), size = 4, hjust = -0.05, color = "black") + geom_tiplab(size = 2, hjust = -2, color = "black")
# 

ggsave("2025.02.18.ncbi_and_novel_new.pdf", width = 150, height = 350, units = "cm", limitsize = F)
#warnings is just size issues, some are very slightly cutoff in the figure, but readable

In addition: Warning messages:
1: Removed 1842 rows containing missing values or values outside the scale range
(`geom_text()`). 
2: Removed 1272 rows containing missing values or values outside the scale range
(`geom_text()`). 


In [36]:
#display pdf
class PDF(object):
  def __init__(self, pdf, size=(200,200)):
    self.pdf = pdf
    self.size = size

  def _repr_html_(self):
    return '<iframe src={0} width={1[0]} height={1[1]}></iframe>'.format(self.pdf, self.size)

  def _repr_latex_(self):
    return r'\includegraphics[width=1.0\textwidth]{{{0}}}'.format(self.pdf)

PDF('2025.02.18.ncbi_and_novel_new.pdf',size=(1000,250))
